*0.4 Deep learning basics*

# Break → Fix

**The situation.** The fine-tuned sentiment model ships behind an API. QA reports that the same review gets 71% positive on one call and 64% on the next. The model file has not changed. Load balancing is blamed, then caching, then the GPU driver.

**The bug.** The serving code never calls `model.eval()`. The model is still in training mode, so dropout is on: on every call a random 10% of activations are zeroed, and the output moves. Batch normalisation (in vision models) would also keep updating its running statistics from live traffic — a model that changes itself in production.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The broken version.** Same input, several calls.

In [2]:
import torch
from torch import nn

torch.manual_seed(0)


class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 2))

    def forward(self, x):
        return self.layers(x)


model = Classifier()  # in production: load_state_dict(...) from the checkpoint
review = torch.randn(1, 64)  # the same input every time

# BREAK: straight to inference, no eval()
outputs = []
for _ in range(5):
    with torch.no_grad():
        outputs.append(torch.softmax(model(review), dim=1)[0, 1].item())


def as_percentages(values):
    result = []
    for value in values:
        result.append(f"{value:.1%}")
    return result


print("BREAK  same input, five calls:", as_percentages(outputs))
print("       model.training =", model.training)
assert len(set(as_percentages(outputs))) > 1

BREAK  same input, five calls: ['65.3%', '56.5%', '64.5%', '65.8%', '59.1%']
       model.training = True


**The fix.** One line, right after loading. And a guard that makes the mistake impossible to ship.

In [3]:
model.eval()  # FIX: dropout off, batch-norm frozen

outputs = []
for _ in range(5):
    with torch.no_grad():
        outputs.append(torch.softmax(model(review), dim=1)[0, 1].item())
print("FIX    same input, five calls:", as_percentages(outputs))


def load_for_serving(model: nn.Module) -> nn.Module:
    model.eval()
    assert not model.training, "model must be in eval mode before serving"
    for parameter in model.parameters():
        parameter.requires_grad_(False)  # nothing in serving should ever update a weight
    return model


served = load_for_serving(Classifier())
trainable = 0
for parameter in served.parameters():
    trainable += int(parameter.requires_grad)
print("guarded:", "training =", served.training, "| trainable parameters =", trainable)
assert len(set(as_percentages(outputs))) == 1

FIX    same input, five calls: ['62.5%', '62.5%', '62.5%', '62.5%', '62.5%']
guarded: training = False | trainable parameters = 0


**Reading the output.** Five different answers, then five identical ones. Nothing changed but the mode.

**How you notice it.** Same input, different outputs; `model.training` is `True` in the serving process; slowly drifting accuracy on models with batch-norm.

**Watch out**
- `eval()` and `no_grad()` are different things: one changes layer behaviour, the other stops gradient bookkeeping. Serving needs both.
- Hugging Face `from_pretrained` returns models in eval mode; a custom `nn.Module` starts in train mode.
- Put the guard in the loader, not in every endpoint.